# Continuous Optimisation

## Imports

In [38]:
import numpy as np
from numpy import linalg as LA
from typing import Tuple, Callable, Optional

## Structural Mechanics

In [45]:
class StructuralMechanics:
    """
    Define the objective function for the gradient descent of the structural mechanics problem.
    """
    
    def __init__(self, n: int = 20):
        # check whether u is in R^n
        assert n > 0, f"n = {n} is not a positive integer."

        self.n: int = n
        self.K: np.ndarray = np.zeros((self.n, self.n))
        self.b: np.ndarray = np.zeros(self.n)

        # b = (0 0 ... 0 1)
        self.b[-1] = 1

        # K = (  2 -1  0 ... 0 )
        #     ( -1  2 -1 ... 0 )
        #     (  0 -1  .     0 )
        #     (  .      .    . )
        #     (  .        . -1 )
        #     (  0  ... 0 -1 2 )
        for i in range(self.n):
            self.K[i, i] = 2
            if i > 0:
                self.K[i - 1, i] = -1
            if i < self.n - 1:
                self.K[i + 1, i] = -1

        eigenvalues = LA.eigvalsh(self.K)
        self.mu: float = float(min(eigenvalues))
        self.nu: float = float(max(eigenvalues))
        
    def _objective(self, u: np.ndarray) -> float:
        """Compute the objective of the function."""
        # E(u) = 1/2 * u^T * K * u - b^T * u
        return 0.5 * u.T @ self.K @ u - self.b.T @ u
    
    def _gradient(self, u: np.ndarray) -> np.ndarray:
        """Compute the gradient of the function."""
        # grad E(u) = K * u - b
        return self.K @ u - self.b

    def _hessian(self) -> np.ndarray:
        """Compute the Hessian of the function."""   
        # hessian E(u) = K     
        return self.K.copy()

    def gradient_descent(
            self, 
            u_0: np.ndarray, 
            alpha: float, 
            epsilon: float = 1e-8, 
            max_iter: int = 10000
            ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Fixed Step Size
        
        Args:
            u_0 (np.ndarray): initial point
            alpha (float): step size
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """ 
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        u_k = u_0.copy()

        for k in range(max_iter):
            grad_k = self._gradient(u_k)
            grad_norm = LA.norm(grad_k)
            
            if grad_norm < epsilon:
                return u_k, k, self._objective(u_k)

            u_k -= alpha * grad_k
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def optimal_step(self, 
        u_k: np.ndarray, 
        d_k: np.ndarray
        ) -> float:
        """Optimal choice of alpha for quadratic functions in given by alpha = -(grad E(u) * d) / (d^T * K * d).

        Args:
            u_k (np.ndarray): current point
            d_k (np.ndarray): descent direction

        Returns:
            float: desired alpha
        """
        grad = self._gradient(u_k)
        numerator = - grad @ d_k
        denominator = d_k.T @ self.K @ d_k
        if abs(denominator) < 1e-12:
            return 0.0
        return float(numerator / denominator)

    def GD_exact_line_search(self,
        u_0: np.ndarray,
        epsilon: float = 1e-8,
        max_iter: int = 10000
        ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Exact Line Search

        Args:
            u_0 (np.ndarray): initial point
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """        
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        u_k = u_0.copy()

        prev_obj = float('inf')
        stagnation_count = 0  # ← ADD TRACKING
        STAGNATION_THRESHOLD = 50  # ← NEW CONSTANT

        for k in range(max_iter):
            grad_k = self._gradient(u_k)
            grad_norm = LA.norm(grad_k)
            
            if grad_norm < epsilon:
                return u_k, k, self._objective(u_k)

            alpha_k = self.optimal_step(u_k, -grad_k)
            u_k -= alpha_k * grad_k

            # ← ADD STAGNATION CHECK
            curr_obj = self._objective(u_k)
            if abs(curr_obj - prev_obj) < 1e-12:
                stagnation_count += 1
                if stagnation_count > STAGNATION_THRESHOLD:
                    print(f"  Warning: Stagnated after {k} iterations, ||grad|| = {grad_norm:.2e}")
                    return u_k, k, curr_obj  # Accept current solution
            else:
                stagnation_count = 0
            prev_obj = curr_obj
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

    def armijo_condition(self,
        u: np.ndarray,
        d: np.ndarray,
        alpha: float,
        c: float
        ) -> bool:
        """Check E(u + alpha * d) <= E(u) + c* alpha * grad E(u)^T * d.

        Args:
            d (np.ndarray): descent direction
            c (float): constant coefficient
            alpha (float): step size

        Returns:
            bool: Is the Armijo condition satisfied.
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha, f"alpha = {alpha} is not > 0."
        step = self._objective(u + alpha * d)
        current = self._objective(u)
        change = c * alpha * self._gradient(u).T @ d
        return (step <= current + change)

    def backtracking(self,
        u_k: np.ndarray,
        d_k: np.ndarray,
        alpha_0: float = 1.0,
        rho: float = 0.5, 
        c: float = 1e-4,
        max_iter: int = 10000
        ) -> float:
        """backtracking line search for sufficiently optimal alpha

        Args:
            u_k (np.ndarray): current point
            d_k (np.ndarray): descent direcion
            alpha_0 (float, optional): initial alpha. Defaults to 1.0.
            rho (float, optional): constant rho. Defaults to 0.5.
            c (float, optional): constant c. Defaults to 1e-4.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Returns:
            float: new sufficient alpha
        """        
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        alpha_new = alpha_0
        for _ in range(max_iter):
            
            if self.armijo_condition(u_k, d_k, alpha_new, c):
                return float(alpha_new)
            
            alpha_new *= rho

        return float(alpha_new)

    def GD_inexact_line_search(self,
        u_0: np.ndarray,
        alpha_0: float = 1.0,
        rho: float = 0.5,
        c: float = 1e-4,
        epsilon: float = 1e-8,
        max_iter: int = 10000
        ) -> Tuple[np.ndarray, int, float]:
        """Gradient Descent with Inexact Line Search

        Args:
            u_0 (np.ndarray): initial point
            alpha_0 (float, optional): initial alpha. Defaults to 1.0.
            rho (float, optional): constant rho. Defaults to 0.5.
            c (float, optional): constant c. Defaults to 1e-4.
            epsilon (float, optional): error margin. Defaults to 1e-8.
            max_iter (int, optional): maximum iterations (for debugging). Defaults to 10000.

        Raises:
            ValueError: Objective function value became infinite.
            RuntimeError: Way too many iterations for the algorithm.

        Returns:
            Tuple[np.ndarray, int, float]: Returns current position, current step number, and current objective value.
        """        
        assert 0 < epsilon, f"epsilon = {epsilon} is not > 0."
        assert (0 < c and c < 1), f"c = {c} is not in (0, 1)."
        assert 0 < alpha_0, f"alpha = {alpha_0} is not > 0."
        assert (0 < rho and rho < 1), f"rho = {rho} is not in (0, 1)."
        
        u_k = u_0.copy()
        alpha_k = alpha_0

        prev_obj = float('inf')
        stagnation_count = 0  # ← ADD TRACKING
        STAGNATION_THRESHOLD = 50  # ← NEW CONSTANT

        for k in range(max_iter):
            grad_k = self._gradient(u_k) # = d_k
            grad_norm = LA.norm(grad_k)
            
            if grad_norm < epsilon:
                return u_k, k, self._objective(u_k)

            alpha_k = self.backtracking(u_k=u_k, d_k=(-grad_k), alpha_0=alpha_k, rho=rho, c=c)
            u_k -= alpha_k * grad_k

            # ← ADD STAGNATION CHECK
            curr_obj = self._objective(u_k)
            if abs(curr_obj - prev_obj) < 1e-12:
                stagnation_count += 1
                if stagnation_count > STAGNATION_THRESHOLD:
                    print(f"  Warning: Stagnated after {k} iterations, ||grad|| = {grad_norm:.2e}")
                    return u_k, k, curr_obj  # Accept current solution
            else:
                stagnation_count = 0
            prev_obj = curr_obj
            
            if not np.isfinite(self._objective(u_k)):
                raise ValueError("Objective became non-finite")
    
        raise RuntimeError(f"Did not converge within {max_iter} iterations")

## Test Suite

In [ ]:
# ============== DOCTESTS ==============
if __name__ == "__main__":
    
    def run_doctests():
        """Run comprehensive doctests for StructuralMechanics class."""
        
        print("=" * 60)
        print("Running Structural Mechanics Doctests")
        print("=" * 60)
        
        # Test 1: Basic initialization
        print("\n[Test 1] Initialization and matrix construction...")
        sm = StructuralMechanics(20)
        assert sm.n == 20, "n should be 20"
        assert sm.b[-1] == 1, "Last element of b should be 1"
        assert sm.b[:-1].sum() == 0, "First n-1 elements of b should be 0"
        print("✓ Matrix K and vector b initialized correctly")
        
        # Test 2: K matrix properties
        print("\n[Test 2] Stiffness matrix properties...")
        K = sm.K
        diag = np.diag(K)
        assert np.allclose(diag, 2), "Diagonal elements should be 2"
        off_diag = np.diag(K, 1)
        assert np.allclose(off_diag, -1), "Super-diagonal should be -1"
        assert np.allclose(K.T, K), "Matrix should be symmetric"
        print("✓ Tridiagonal structure verified")
        
        # Test 3: Objective function calculation
        print("\n[Test 3] Objective function...")
        u_test = np.ones(20)
        manual_energy = 0.5 * u_test.T @ sm.K @ u_test - sm.b.T @ u_test
        assert np.isclose(sm._objective(u_test), manual_energy), "Energy calculation mismatch"
        print(f"✓ Objective = {sm._objective(u_test):.6f}")
        
        # Test 4: Gradient calculation
        print("\n[Test 4] Gradient computation...")
        grad_manual = K @ u_test - sm.b
        assert np.allclose(sm._gradient(u_test), grad_manual), "Gradient mismatch"
        print("✓ Gradient verified analytically")
        
        # Test 5: Eigenvalue properties (convexity check)
        print("\n[Test 5] Convexity via eigenvalues...")
        assert sm.mu > 0, "Problem should be strictly convex (mu > 0)"
        cond_number = sm.nu / sm.mu
        print(f"  μ_min = {sm.mu:.6f}, ν_max = {sm.nu:.6f}, κ = {cond_number:.2f}")
        print("✓ Problem is well-conditioned and convex")
        
        # Test 6: Fixed-step gradient descent
        print("\n[Test 6] Fixed-step gradient descent...")
        u0_small = np.zeros(10)
        sm_small = StructuralMechanics(10)
        optimal_alpha = 2 / (sm_small.nu + sm_small.mu)
        u_opt, iter_opt, obj_opt = sm_small.gradient_descent(
            u0_small, alpha=optimal_alpha, epsilon=1e-10
        )
        print(f"  Converged in {iter_opt} iterations, final obj = {obj_opt:.8f}")
        print("✓ Fixed-step descent works")
        
        # Test 7: Exact line search (UNCOMMENTED - was disabled)
        print("\n[Test 7] Exact line search...")
        u_exact, iter_exact, obj_exact = sm_small.GD_exact_line_search(
            u_0=u0_small, epsilon=1e-10
        )
        print(f"  Converged in {iter_exact} iterations, final obj = {obj_exact:.8f}")
        assert obj_exact < obj_opt or abs(obj_exact - obj_opt) < 1e-10, \
            "Exact search should perform at least as well as optimal fixed step"
        print("✓ Exact line search converged faster or equal to fixed step")
        
        # Test 8: Inexact line search (backtracking)
        print("\n[Test 8] Inexact line search (backtracking)...")
        u_inexact, iter_inexact, obj_inexact = sm_small.GD_inexact_line_search(
            u_0=u0_small, alpha_0=1.0, epsilon=1e-10
        )
        print(f"  Converged in {iter_inexact} iterations, final obj = {obj_inexact:.8f}")
        print("✓ Backtracking line search worked")
        
        # Test 9: Compare all three methods (UNCOMMENTED - now shows all 3)
        print("\n[Test 9] Performance comparison...")
        print(f"  {'Method':<25} {'Iterations':>12} {'Final Obj':>15}")
        print(f"  {'-'*25} {'-'*12} {'-'*15}")
        print(f"  {'Fixed Step':<25} {iter_opt:>12} {obj_opt:>15.8f}")
        print(f"  {'Exact Line Search':<25} {iter_exact:>12} {obj_exact:>15.8f}")  # UNCOMMENTED
        print(f"  {'Inexact (Backtracking)':<25} {iter_inexact:>12} {obj_inexact:>15.8f}")
        
        # Test 10: Analytical solution check
        print("\n[Test 10] Compare with analytical solution K⁻¹b...")
        u_analytical = LA.solve(sm_small.K, sm_small.b)
        print(f"the analytical value for the solution is {u_analytical}\n")
        u_converged, _, _ = sm_small.GD_exact_line_search(u_0=np.zeros(10), epsilon=1e-12)
        error = LA.norm(u_converged - u_analytical)
        print(f"  ||u_converged - u_analytical|| = {error:.2e}")
        assert error < 1e-5, f"Convergence error too large: {error}"
        print("✓ Converges to analytical minimum")
                
        print("\n" + "=" * 60)
        print("All tests passed! ✓")
        print("=" * 60)
    
    run_doctests()

Running StructuralMechanics Doctests

[Test 1] Initialization and matrix construction...
✓ Matrix K and vector b initialized correctly

[Test 2] Stiffness matrix properties...
✓ Tridiagonal structure verified

[Test 3] Objective function...
✓ Objective = 0.000000

[Test 4] Gradient computation...
✓ Gradient verified analytically

[Test 5] Convexity via eigenvalues...
  μ_min = 0.022338, ν_max = 3.977662, κ = 178.06
✓ Problem is well-conditioned and convex

[Test 6] Fixed-step gradient descent...
  Converged in 514 iterations, final obj = -0.45454545
✓ Fixed-step descent works

[Test 7] Exact line search...
  Converged in 325 iterations, final obj = -0.45454545
✓ Exact line search converged faster or equal to fixed step

[Test 8] Inexact line search (backtracking)...
  Converged in 325 iterations, final obj = -0.45454545
✓ Backtracking line search worked

[Test 9] Performance comparison...
  Method                      Iterations       Final Obj
  ------------------------- ------------ 